In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ============================================================
# FILE PATHS
# ============================================================

BASE_DATASET = "impact_dataset.csv"
FEEDBACK_LOG = "astram_retraining_log.csv"   # <-- your feedback file
MODEL_OUTPUT = "traffic_impact_predictor.pkl"

# ============================================================
# MODEL CONFIG
# ============================================================

FEATURE_COLS = [
    "event_cause",
    "priority",
    "requires_road_closure",
    "event_type",
    "junction"
]

TARGET_COL = "impact_score"

# Optional: make feedback influence stronger than old historical data
FEEDBACK_WEIGHT = 5   # try 3 to 8; 5 is a good demo value

# ============================================================
# LOAD BASE DATASET
# ============================================================

if not os.path.exists(BASE_DATASET):
    raise FileNotFoundError(f"Base dataset not found: {BASE_DATASET}")

base_df = pd.read_csv(BASE_DATASET)

required_base_cols = FEATURE_COLS + [TARGET_COL]
missing_base = [c for c in required_base_cols if c not in base_df.columns]
if missing_base:
    raise ValueError(f"Base dataset is missing required columns: {missing_base}")

base_df = base_df[required_base_cols].copy()

# Clean base dataset
base_df["event_cause"] = base_df["event_cause"].astype(str).str.strip()
base_df["priority"] = base_df["priority"].astype(str).str.strip()
base_df["event_type"] = base_df["event_type"].astype(str).str.strip().str.lower()
base_df["junction"] = base_df["junction"].astype(str).str.strip()
base_df["requires_road_closure"] = base_df["requires_road_closure"].astype(str).str.strip().str.lower().map({
    "true": True,
    "false": False
}).fillna(base_df["requires_road_closure"])
base_df[TARGET_COL] = pd.to_numeric(base_df[TARGET_COL], errors="coerce")

base_df = base_df.dropna(subset=FEATURE_COLS + [TARGET_COL])

print(f"Loaded base dataset rows: {len(base_df)}")

# ============================================================
# LOAD FEEDBACK LOG
# ============================================================

feedback_training_df = pd.DataFrame(columns=FEATURE_COLS + [TARGET_COL])

if os.path.exists(FEEDBACK_LOG):
    try:
        fb = pd.read_csv(FEEDBACK_LOG)

        required_feedback_cols = [
            "event_cause",
            "priority",
            "requires_road_closure",
            "event_type",
            "junction",
            "observed_field_impact_score"
        ]

        missing_fb = [c for c in required_feedback_cols if c not in fb.columns]

        if missing_fb:
            print("Feedback log found, but required columns are missing:")
            print(missing_fb)

        elif len(fb) == 0:
            print("Feedback log is empty. No feedback rows used.")

        else:
            # Keep only needed columns
            feedback_training_df = fb[
                [
                    "event_cause",
                    "priority",
                    "requires_road_closure",
                    "event_type",
                    "junction",
                    "observed_field_impact_score"
                ]
            ].copy()

            # Rename target column to match base dataset
            feedback_training_df.rename(
                columns={"observed_field_impact_score": "impact_score"},
                inplace=True
            )

            # Clean feedback rows
            feedback_training_df["event_cause"] = feedback_training_df["event_cause"].astype(str).str.strip()
            feedback_training_df["priority"] = feedback_training_df["priority"].astype(str).str.strip()
            feedback_training_df["event_type"] = feedback_training_df["event_type"].astype(str).str.strip().str.lower()
            feedback_training_df["junction"] = feedback_training_df["junction"].astype(str).str.strip()

            feedback_training_df["requires_road_closure"] = (
                feedback_training_df["requires_road_closure"]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({"true": True, "false": False})
            )

            feedback_training_df["impact_score"] = pd.to_numeric(
                feedback_training_df["impact_score"],
                errors="coerce"
            )

            feedback_training_df = feedback_training_df.dropna(
                subset=FEATURE_COLS + ["impact_score"]
            )

            print(f"Loaded feedback rows before weighting: {len(feedback_training_df)}")

            # ====================================================
            # WEIGHT FEEDBACK ROWS
            # ====================================================
            if len(feedback_training_df) > 0 and FEEDBACK_WEIGHT > 1:
                feedback_training_df = pd.concat(
                    [feedback_training_df] * FEEDBACK_WEIGHT,
                    ignore_index=True
                )
                print(f"Feedback rows after weighting x{FEEDBACK_WEIGHT}: {len(feedback_training_df)}")

    except Exception as e:
        print(f"Could not read feedback log. Proceeding with base dataset only. Error: {e}")

else:
    print("No feedback log found. Training only on base dataset.")

# ============================================================
# COMBINE BASE + FEEDBACK
# ============================================================

combined_df = pd.concat([base_df, feedback_training_df], ignore_index=True)
combined_df = combined_df.dropna(subset=FEATURE_COLS + [TARGET_COL])
combined_df = combined_df.drop_duplicates()

print(f"Final training rows: {len(combined_df)}")
print(f"Feedback rows actually used: {len(feedback_training_df)}")

# ============================================================
# SPLIT DATA
# ============================================================

X = combined_df[FEATURE_COLS].copy()
y = combined_df[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ============================================================
# PREPROCESSOR
# ============================================================

categorical_cols = [
    "event_cause",
    "priority",
    "event_type",
    "junction"
]

binary_cols = ["requires_road_closure"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("bin", "passthrough", binary_cols)
    ]
)

# ============================================================
# MODEL
# ============================================================

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_split=3,
        min_samples_leaf=1,
        random_state=42
    ))
])

# ============================================================
# TRAIN
# ============================================================

model.fit(X_train, y_train)

# ============================================================
# EVALUATE
# ============================================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n================ RETRAINING COMPLETE ================")
print(f"MAE : {mae:.3f}")
print(f"R²  : {r2:.3f}")

# ============================================================
# SAVE MODEL
# ============================================================

joblib.dump(model, MODEL_OUTPUT)
print(f"\nUpdated model saved as: {MODEL_OUTPUT}")

Loaded base dataset rows: 8173
Loaded feedback rows before weighting: 1
Feedback rows after weighting x5: 5
Final training rows: 895
Feedback rows actually used: 5

================ RETRAINING COMPLETE ================
MAE : 0.700
R²  : 0.943

Updated model saved as: traffic_impact_predictor.pkl
